In [4]:
import pandas as pd

In [5]:
customer=pd.read_csv("customers.csv")
orders=pd.read_csv("orders.csv")
product=pd.read_csv("products.csv")

### Load and inspect raw CSVs in Colab


In [6]:
print("Rows and Columns in Customer data",customer.shape)
print("--"*13)
print("Rows and Columns in Orders data",orders.shape)
print("--"*13)
print("Rows and Columns in Product data",product.shape)

Rows and Columns in Customer data (45, 6)
--------------------------
Rows and Columns in Orders data (180, 9)
--------------------------
Rows and Columns in Product data (16, 4)


### Standardize payment_method casing using Pandas .str accessor

In [7]:
print("Distinct Raw Values Of Payment Method",list(orders["payment_method"].unique()))

Distinct Raw Values Of Payment Method ['upi', 'UPI', 'COD', 'Card', 'cod', 'CARD', 'card']


In [22]:

orders["payment_method"]=orders["payment_method"].str.strip().str.upper()

print("Standardized Raw Values Counts Of Paymnet Method",orders["payment_method"].value_counts())

Standardized Raw Values Counts Of Paymnet Method payment_method
CARD    70
UPI     55
COD     55
Name: count, dtype: int64


### Remove duplicate order records based on natural keys

In [9]:
duplicated_orders = orders[orders.duplicated(subset=['customer_id', 'product_id', 'order_date', 'quantity',
       'discount_pct', 'payment_method', 'rating', 'returned'], keep='first')]
print("Duplicate Order Id:",list(duplicated_orders["order_id"]))
orders_clean=orders.drop_duplicates(subset=['customer_id', 'product_id', 'order_date', 'quantity',
       'discount_pct', 'payment_method', 'rating', 'returned'], keep='first')
print("Shape of Cleaned Data",orders_clean.shape)

Duplicate Order Id: ['O0176', 'O0177', 'O0178', 'O0179', 'O0180']
Shape of Cleaned Data (175, 9)


### Impute missing discount_pct (zero) and rating (median)

In [10]:
print("Missing Values in discount_pct Column On Orders",orders_clean["discount_pct"].isnull().sum())
print("Missing Values in rating Column  On Orders",orders_clean["rating"].isnull().sum())

Missing Values in discount_pct Column On Orders 12
Missing Values in rating Column  On Orders 15


In [11]:
orders_clean["discount_pct"]=orders_clean["discount_pct"].fillna(0)

print("Median of Rating Column On Orders",orders_clean["rating"].median())

orders_clean["rating"]=orders_clean["rating"].fillna(orders_clean["rating"].median())

print(orders_clean[["discount_pct","rating"]].isnull().sum())

Median of Rating Column On Orders 3.0
discount_pct    0
rating          0
dtype: int64


/tmp/ipykernel_808/1557193456.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  orders_clean["discount_pct"]=orders_clean["discount_pct"].fillna(0)
/tmp/ipykernel_808/1557193456.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  orders_clean["rating"]=orders_clean["rating"].fillna(orders_clean["rating"].median())


### Merge DataFrames, compute revenue, and output reconciliation note

In [12]:
merged_df = orders_clean.merge(product, on='product_id')
merged_df = merged_df.merge(customer, on='customer_id', how='inner')

print(f"Original orders shape: {orders.shape}")
print(f"Merged DataFrame shape: {merged_df.shape}")

Original orders shape: (180, 9)
Merged DataFrame shape: (175, 17)


In [13]:
merged_df['order_value'] = merged_df['quantity'] * merged_df['price'] * (1 - merged_df['discount_pct'] / 100.0)

total_order_value = merged_df['order_value'].sum()
print(f"Total Calculated Order Value from Merged Table: {total_order_value:,.2f}")


Total Calculated Order Value from Merged Table: 97,358.30


In [14]:
merged_duplicated_orders=duplicated_orders.merge(product,on='product_id')
merged_duplicated_orders['order_value'] = merged_duplicated_orders['quantity'] * merged_duplicated_orders['price'] * (1 - merged_duplicated_orders['discount_pct'] / 100.0)
total_duplicated_order_value=merged_duplicated_orders['order_value'].sum()

print(f"Total Calculated Duplicate Order Value from Merged Table: {total_duplicated_order_value:,.2f}")


Total Calculated Duplicate Order Value from Merged Table: 2,501.90


In [15]:
reconciliation_note = (
    "RECONCILIATION NOTE: The difference between the original and cleaned dataset order value is "
    "strictly due to the removal of the 5 duplicate rows in Task 3. The combined order value of these "
    f"5 duplicate rows is exactly ₹{total_duplicated_order_value:,.2f}. Imputing the missing values "
    "(such as filling NaN in discount_pct with 0 and rating with its median) did not affect any existing "
    "order values because missing discounts were already mathematically calculated as zero-effect in both stages, "
    "and ratings are not used in calculating the financial value of the orders. Thus, the exact financial "
    "variance of the dataset is fully and exclusively accounted for by the duplicate elimination."
)
print(reconciliation_note)

RECONCILIATION NOTE: The difference between the original and cleaned dataset order value is strictly due to the removal of the 5 duplicate rows in Task 3. The combined order value of these 5 duplicate rows is exactly ₹2,501.90. Imputing the missing values (such as filling NaN in discount_pct with 0 and rating with its median) did not affect any existing order values because missing discounts were already mathematically calculated as zero-effect in both stages, and ratings are not used in calculating the financial value of the orders. Thus, the exact financial variance of the dataset is fully and exclusively accounted for by the duplicate elimination.


### Perform IQR outlier detection on order quantity

In [16]:
Q1=merged_df['quantity'].quantile(0.25)
Q3=merged_df['quantity'].quantile(0.75)
IQR=Q3-Q1
lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR

outliers= merged_df[(merged_df['quantity']<lower) | (merged_df['quantity']>upper)]

merged_df["is_outlier"]=False
merged_df.loc[outliers.index,"is_outlier"]=True


### Test hypothesis — evaluate return rates by payment_method

In [23]:
print(round(merged_df.groupby('payment_method')['returned'].agg(['count','mean'])*100,1))

                count  mean
payment_method             
CARD             6800  14.7
COD              5400  44.4
UPI              5300  18.9


### Perform multi-level segmentation on payment_method and city_tier

In [18]:
round(merged_df.groupby(['payment_method','city_tier'])['returned'].agg(['mean'])*100,2)

mean
payment_method city_tier       
CARD           1           8.89
               2          26.09
COD            1          37.50
               2          54.55
UPI            1          19.35
               2          18.18

### Compute correlation matrix and evaluate taught strength bands

In [19]:
cols = ['rating', 'returned', 'discount_pct', 'quantity']
corr_matrix = merged_df[cols].corr()

print("--- Correlation Matrix ---")
display(corr_matrix.round(4))
print("\n--- Pairwise Correlation Analysis ---")

def evaluate_correlation(r):
    abs_r = abs(r)
    if abs_r < 0.2:
        return "negligible"
    elif abs_r < 0.4:
        return "weak"
    elif abs_r < 0.7:
        return "moderate"
    else:
        return "strong"

pairs = [
    ('rating', 'returned'),
    ('rating', 'discount_pct'),
    ('rating', 'quantity'),
    ('returned', 'discount_pct'),
    ('returned', 'quantity'),
    ('discount_pct', 'quantity')
]

for var1, var2 in pairs:
    r_val = corr_matrix.loc[var1, var2]
    band = evaluate_correlation(r_val)
    print(f"{var1} vs {var2}: r = {r_val:.4f} -> Strength Band: {band.upper()}")

discount_returned_corr = corr_matrix.loc['discount_pct', 'returned']
print("\n--- Hypothesis Evaluation ---")
print(f"Hypothesis 'higher discounts reduce returns': BUSTED.")
print(f"Reason: The correlation between discount_pct and returned is {discount_returned_corr.round(2)}, which falls squarely in the NEGLIGIBLE (|r| < 0.2) band.")

--- Correlation Matrix ---


,rating,returned,discount_pct,quantity
rating,1.0000,-0.1015,0.0427,-0.0980
returned,-0.1015,1.0000,-0.0880,0.0406
discount_pct,0.0427,-0.0880,1.0000,-0.1704
quantity,-0.0980,0.0406,-0.1704,1.0000



--- Pairwise Correlation Analysis ---
rating vs returned: r = -0.1015 -> Strength Band: NEGLIGIBLE
rating vs discount_pct: r = 0.0427 -> Strength Band: NEGLIGIBLE
rating vs quantity: r = -0.0980 -> Strength Band: NEGLIGIBLE
returned vs discount_pct: r = -0.0880 -> Strength Band: NEGLIGIBLE
returned vs quantity: r = 0.0406 -> Strength Band: NEGLIGIBLE
discount_pct vs quantity: r = -0.1704 -> Strength Band: NEGLIGIBLE

--- Hypothesis Evaluation ---
Hypothesis 'higher discounts reduce returns': BUSTED.
Reason: The correlation between discount_pct and returned is -0.09, which falls squarely in the NEGLIGIBLE (|r| < 0.2) band.


### Compute raw vs. outlier-corrected monthly revenue time series

In [20]:
merged_df['order_date']=pd.to_datetime(merged_df['order_date'])
# merged_df['year']=merged_df['order_date'].dt.year
# merged_df['month']=merged_df['order_date'].dt.month
merged_df['year-month']=merged_df['order_date'].dt.to_period('M')

merged_df_order_values_with_outliers=merged_df.groupby('year-month')['order_value'].sum()
merged_df_order_values_without_outliers=merged_df[merged_df['is_outlier']==False].groupby('year-month')['order_value'].sum()
print("InCluding Outliers")
print(merged_df_order_values_with_outliers)
print("Excluding Outliers")
print(merged_df_order_values_without_outliers)

InCluding Outliers
year-month
2026-01    29582.1
2026-02    13195.5
2026-03    20318.9
2026-04     9495.3
2026-05    13151.1
2026-06    11615.4
Freq: M, Name: order_value, dtype: float64
Excluding Outliers
year-month
2026-01    11637.1
2026-02    13195.5
2026-03    20318.9
2026-04     9495.3
2026-05    13151.1
2026-06    11615.4
Freq: M, Name: order_value, dtype: float64


### storing the findings in findings.json

In [21]:
import json

raw_total_revenue = total_order_value + total_duplicated_order_value

return_rates = merged_df.groupby('payment_method')['returned'].mean() * 100
return_rate_by_payment = {
    "COD": round(return_rates.get('COD', 0), 1),
    "CARD": round(return_rates.get('CARD', 0), 1),
    "UPI": round(return_rates.get('UPI', 0), 1)
}

risk_segment = merged_df.groupby(['payment_method', 'city_tier'])['returned'].mean() * 100
highest_risk = risk_segment.idxmax()
highest_risk_rate = risk_segment.max()

highest_risk_segment = {
    "payment_method": highest_risk[0],
    "city_tier": int(highest_risk[1]),
    "return_rate_pct": round(highest_risk_rate, 1)
}

true_peak_month_str = str(merged_df_order_values_without_outliers.idxmax())
true_peak_revenue = merged_df_order_values_without_outliers.max()

apparent_revenue_jan = merged_df_order_values_with_outliers['2026-01']
corrected_revenue_jan = merged_df_order_values_without_outliers['2026-01']

metrics_data = {
    "cleaned_total_revenue_inr": round(float(total_order_value), 2),
    "raw_total_revenue_inr": round(float(raw_total_revenue), 2),
    "duplicate_reconciliation_delta_inr": round(float(total_duplicated_order_value), 2),
    "return_rate_by_payment": return_rate_by_payment,
    "highest_risk_segment": highest_risk_segment,
    "true_peak_month": {
        "month": true_peak_month_str,
        "revenue_inr": round(float(true_peak_revenue), 2)
    },
    "outlier_inflated_month": {
        "month": "2026-01",
        "apparent_revenue_inr": round(float(apparent_revenue_jan), 2),
        "corrected_revenue_inr": round(float(corrected_revenue_jan), 2)
    }
}

file_path = 'findings.json'
with open(file_path, 'w') as f:
    json.dump(metrics_data, f, indent=4)


In [27]:
cleaned_file_path = 'cleaned_orders_data.csv'
merged_df.to_csv(cleaned_file_path, index=False)
print(f"Cleaned data has been successfully saved to {cleaned_file_path}")

Cleaned data has been successfully saved to cleaned_orders_data.csv
